<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Autonomous Traders</h2>
            <span style="color:#ff7800;">An equity trading simulation to illustrate autonomous agents powered by tools and resources from MCP servers.
            </span>
        </td>
    </tr>
</table>

### Week 6 Day 4

And now - introducing the Capstone project:


# Autonomous Traders

An equity trading simulation, with 4 Traders and a Researcher, powered by a slew of MCP servers with tools & resources:

1. Our home-made Accounts MCP server (written by our engineering team!)
2. Fetch (get webpage via a local headless browser)
3. Memory
4. Brave Search
5. Financial data

And a resource to read information about the trader's account, and their investment strategy.

The goal of today's lab is to make a new python module, `traders.py` that will manage a single trader on our trading floor.

We will experiment and explore in the lab, and then migrate to a python module when we're ready.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">One more time --</h2>
            <span style="color:#ff7800;">Please do not use this for actual trading decisions!!
            </span>
        </td>
    </tr>
</table>

In [1]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

load_dotenv(override=True)

True

### Let's start by gathering the MCP params for our trader

In [2]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(is_paid_polygon)
print(is_realtime_polygon)

False
False


In [3]:
# if using paid version, we will have a whole lots of MCP servers/tools
if is_paid_polygon or is_realtime_polygon:
    market_mcp = {"command": "uvx","args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@master", "mcp_polygon"], "env": {"POLYGON_API_KEY": polygon_api_key}}
# else, we will use the market_server.py file with caching
else:
    market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

### And now for our researcher

In [4]:
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

# fetch is a tool that can fetch data for a specific url
# brave search is a tool that can search the web for a specific query
# we use brave search not serper because it has an official MCP server maintained by Anthropic/ModelContextProtocol
# Serper would require custom MCP server implementation or wrapper, adding complexity.
researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}
]

### Now create the MCPServerStdio for each

In [5]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

### Now let's make a Researcher Agent to do market research

And turn it into a tool - remember how this works for OpenAI Agents SDK, and the difference with handoffs?

In [6]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""You are a financial researcher. You are able to search the web for interesting financial news,
look for possible trading opportunities, and help with research.
Based on the request, you carry out necessary research and respond with your findings.
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
If there isn't a specific request, then just respond with investment opportunities based on searching latest news.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    # here we prefer to pass the datetime to the agent
    # so that agent don't need to deal with another tool to get the datetime
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
    )
    return researcher

In [7]:
# the researcher will be available to the trader
# so that trader can use the researcher as a tool
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="This tool researches online for news and opportunities, \
                either based on your specific request to look into a certain stock, \
                or generally for notable financial news and opportunities. \
                Describe what kind of research you're looking for."
        )

In [8]:
import os
from langfuse import Langfuse
from langfuse import observe, get_client
from langfuse.openai import openai
from contextlib import asynccontextmanager
import json
from datetime import datetime

load_dotenv(override=True)

langfuse = get_client()


In [9]:
research_question = "What's the latest news on Amazon?"

# this for loop is to connect to the MCP servers
# this is another way of doing it, avoiding using nested with statements
for server in researcher_mcp_servers:
    await server.connect()
# get the researcher agent itself
# the researcher agent can use the MCP servers for brave search and fetch
researcher = await get_researcher(researcher_mcp_servers)
# with trace("Researcher"):
with langfuse.start_as_current_span(name="Researcher") as span:
    # max_turns is the maximum number of turns/tool calls the agent can take
    # setting it to 30 to get a deeper research (to conduct multiple searches/tool calling)
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))



[non-fatal] Tracing client error 403: {
  "error": {
    "message": "Traces ingestion is not permitted for zero data retention organizations.",
    "type": "invalid_request_error",
    "param": null,
    "code": "zdr_forbidden"
  }
}
[non-fatal] Tracing client error 403: {
  "error": {
    "message": "Traces ingestion is not permitted for zero data retention organizations.",
    "type": "invalid_request_error",
    "param": null,
    "code": "zdr_forbidden"
  }
}


Here are the latest news highlights about Amazon in July 2025:

1. Amazon Prime Day 2025 is scheduled for July 8-11, expanding to four days instead of the usual two. This event will feature deep discounts across more than 35 categories and will be available in 26 countries, including new locations like Ireland and Colombia. A new "Today's Big Deals" feature will launch during Prime Day, dropping themed daily deals from top brands such as Samsung, Kiehl's, and Levi's.

2. Amazon's first quarter results showed net sales increased 9% to $155.7 billion compared with the first quarter of 2024. Excluding an unfavorable $1.4 billion impact from foreign exchange rate changes, net sales increased by 10%.

3. Amazon continues to enhance Prime delivery speeds, reaching more smaller cities, towns, and rural areas.

4. Amazon is actively growing its satellite fleet with the successful second launch of its Kuiper mission, aimed at expanding internet access.

5. On Amazon Prime Video, several new shows and movies are featured in July 2025, including titles like "Heads of State" starring John Cena and Idris Elba, and the "Bosch" spin-off "Ballard."

If you want more specific updates or information on any particular aspect of Amazon, let me know!

[non-fatal] Tracing client error 403: {
  "error": {
    "message": "Traces ingestion is not permitted for zero data retention organizations.",
    "type": "invalid_request_error",
    "param": null,
    "code": "zdr_forbidden"
  }
}
[non-fatal] Tracing client error 403: {
  "error": {
    "message": "Traces ingestion is not permitted for zero data retention organizations.",
    "type": "invalid_request_error",
    "param": null,
    "code": "zdr_forbidden"
  }
}
[non-fatal] Tracing client error 403: {
  "error": {
    "message": "Traces ingestion is not permitted for zero data retention organizations.",
    "type": "invalid_request_error",
    "param": null,
    "code": "zdr_forbidden"
  }
}
[non-fatal] Tracing client error 403: {
  "error": {
    "message": "Traces ingestion is not permitted for zero data retention organizations.",
    "type": "invalid_request_error",
    "param": null,
    "code": "zdr_forbidden"
  }
}
[non-fatal] Tracing client error 403: {
  "error": {
    "messag

### Look at the trace

https://platform.openai.com/traces

In [10]:
ed_initial_strategy = "You are a day trader that aggressively buys and sells shares based on news and market conditions."
Account.get("Ed").reset(ed_initial_strategy)

display(Markdown(await read_accounts_resource("Ed")))
display(Markdown(await read_strategy_resource("Ed")))

{"name": "ed", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2025-07-06 16:32:49", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

You are a day trader that aggressively buys and sells shares based on news and market conditions.

### And now - to create our Trader Agent

In [12]:
agent_name = "Ed"

# Using MCP Servers to read resources
# call the MCP client, then calls the mcp server to read the resources
# the resources are just text shoved into the prompt
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
You are a trader that manages a portfolio of shares. Your name is {agent_name} and your account is under your name, {agent_name}.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
{strategy}
Your current holdings and balance is:
{account_details}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.
Please make use of these tools to manage your portfolio. Carry out trades as you see fit; do not wait for instructions or ask for confirmation.
"""

prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [13]:
print(instructions)


You are a trader that manages a portfolio of shares. Your name is Ed and your account is under your name, Ed.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
You are a day trader that aggressively buys and sells shares based on news and market conditions.
Your current holdings and balance is:
{"name": "ed", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2025-07-06 16:32:49", 10000.0], ["2025-07-06 16:33:09", 10000.0], ["2025-07-06 20:15:14", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, re

### And to run our Trader

In [14]:
for server in mcp_servers:
    await server.connect()

# turn researcher agent into a tool
researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    # trader has access to the researcher tool
    tools=[researcher_tool],
    # trader has access to the MCP servers (which includes the account server, push server, and market/polygon server)
    mcp_servers=trader_mcp_servers,
    model="gpt-4o-mini",
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

### Summary of Actions Taken:

1. **Market Research**: I investigated the latest market trends and identified stocks with high volatility and significant movements suitable for day trading.

2. **Stock Transactions**:
   - **Purchased**:
     - **Quantum Computing Inc. (QUBT)**: 200 shares at $21.17. Rationale: High volatility and significant dollar movement.
     - **BigBear.ai Holdings Inc. (BBAI)**: 100 shares at $7.77. Rationale: Significant daily range and volume.
     - **Sunrun Inc. (RUN)**: 150 shares at $10.52. Rationale: Trending upward with volatility.
     - **Super Micro Computer Inc. (SMCI)**: Attempted but did not have sufficient funds after making the above purchases.

3. **Adjustment**:
   - **Sold**: 
     - 100 shares of QUBT at $21.09. Rationale: Taking profits while holding some shares.

### Current Portfolio Overview:
- **Holdings**:
  - QUBT: 100 shares
  - BBAI: 100 shares
  - RUN: 150 shares

- **Account Balance**: $5,519.62
- **Total Portfolio Value**: $9,982.62
- **Total Profit/Loss**: -$17.38

### Next Steps:
I will continue to monitor the market and look for further opportunities to capitalize on trades based on real-time news and price movements. Would you like to explore more stocks or adjust your strategy?

### Then go and look at the trace

http://platform.openai.com/traces


In [15]:
# And let's look at the results of the trading

await read_accounts_resource(agent_name)

'{"name": "ed", "balance": 5519.622, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"QUBT": 100, "BBAI": 100, "RUN": 150}, "transactions": [{"symbol": "QUBT", "quantity": 200, "price": 21.172259999999998, "timestamp": "2025-07-06 20:16:32", "rationale": "High volatility and significant dollar movement make Quantum Computing a strong day trading opportunity."}, {"symbol": "BBAI", "quantity": 100, "price": 7.7655, "timestamp": "2025-07-06 20:16:32", "rationale": "BigBear.ai has significant daily range and volume, ideal for capturing short-term gains."}, {"symbol": "RUN", "quantity": 150, "price": 10.521, "timestamp": "2025-07-06 20:16:32", "rationale": "Sunrun Inc. is trending upward and has good volatility for day trading."}, {"symbol": "QUBT", "quantity": -100, "price": 21.08774, "timestamp": "2025-07-06 20:16:33", "rationale": "Taking profits on a volatile stock while still holding some."}], "portfolio_valu

### Now it's time to review the Python module made from this:

`mcp_params.py` is where the MCP servers are specified. You'll notice I've brought in some familiar friends: memory and push notifications!

`templates.py` is where the instructions and messages are set up (i.e. the System prompts and User prompts)

`traders.py` brings it all together.

You'll notice I've done something a bit fancy with code like this:

```
async with AsyncExitStack() as stack:
    mcp_servers = [await stack.enter_async_context(MCPServerStdio(params)) for params in mcp_server_params]
```

This is just a tidy way to combine our "with" statements (known as context managers) so that we don't need to do something ugly like this:

```
async with MCPServerStdio(params=params1) as mcp_server1:
    async with MCPServerStdio(params=params2) as mcp_server2:
        async with MCPServerStdio(params=params3) as mcp_server3:
            mcp_servers = [mcp_server1, mcp_server2, mcp_server3]
```

But it's equivalent.


In [16]:
from traders import Trader


In [17]:
trader = Trader("Ed")

In [18]:
await trader.run()

In [19]:
await read_accounts_resource("Ed")

'{"name": "ed", "balance": 6817.022, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"QUBT": 100, "RUN": 100}, "transactions": [{"symbol": "QUBT", "quantity": 200, "price": 21.172259999999998, "timestamp": "2025-07-06 20:16:32", "rationale": "High volatility and significant dollar movement make Quantum Computing a strong day trading opportunity."}, {"symbol": "BBAI", "quantity": 100, "price": 7.7655, "timestamp": "2025-07-06 20:16:32", "rationale": "BigBear.ai has significant daily range and volume, ideal for capturing short-term gains."}, {"symbol": "RUN", "quantity": 150, "price": 10.521, "timestamp": "2025-07-06 20:16:32", "rationale": "Sunrun Inc. is trending upward and has good volatility for day trading."}, {"symbol": "QUBT", "quantity": -100, "price": 21.08774, "timestamp": "2025-07-06 20:16:33", "rationale": "Taking profits on a volatile stock while still holding some."}, {"symbol": "BBAI", "quantity"

### Now look at the trace

https://platform.openai.com/traces

### How many tools did we use in total?

In [ ]:
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params

all_params = trader_mcp_server_params + researcher_mcp_server_params("ed")

count = 0
for each_params in all_params:
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
print(f"We have {len(all_params)} MCP servers, and {count} tools")